# 02 · Does the context graph beat grep? — and how to run an honest eval

In **nb01** we built an L3 code graph. Now the honest question: **does it actually beat plain `grep`?**

Two agents — **same model, same budget** — only the *tools* differ:

| | tools | level |
|---|---|---|
| **graph agent** | curated Neo4j traversal (`search`, `blast_radius`, `implementors`, …) | L3 |
| **grep agent** | `grep` / `glob` / `read_file` over the raw source | L1 |

We measure **outcomes**, we name **where it loses**, and we score **deterministically — no LLM judge.**
That last choice isn't just thrift: the headline result turns on *determinism*, and **you can't measure
determinism with a non-deterministic ruler.**

> Everything below renders from a **committed cache** (one canonical run, `k=5`) — it runs with no keys and
> no Neo4j. Set `SAMPLE_SIZE > 0` to watch a few questions run live.
>
> *Cache generated with `openrouter:openai/gpt-5.4-mini`. The optional live sample uses your `AGENT_MODEL` via OpenRouter — numbers may differ; the cache is the canonical result.*\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bcallender/agent-context-workshop/blob/main/notebooks/02_does_the_graph_beat_grep.ipynb)  ·  *zero-install: click, then run the cells top to bottom.*

In [1]:
# Colab bootstrap — no-op locally. In Colab: clone the repo, install it, hydrate the data.
import sys, os
if "google.colab" in sys.modules:
    import subprocess
    REPO = "/content/agent-context-workshop"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/bcallender/agent-context-workshop.git", REPO], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    subprocess.run(["bash", "scripts/setup_data.sh"], check=True)
    print("Colab ready — cloned, installed, data hydrated.")

In [2]:
# setup + presentation (collapsed) — loads the committed cache; no keys/Neo4j needed to render.
from IPython.display import HTML, display
display(HTML("""<style>
.jp-RenderedHTMLCommon{font-size:1.15rem;line-height:1.6}.jp-RenderedHTMLCommon h1{font-size:2rem}
.jp-RenderedHTMLCommon h2{font-size:1.45rem}.jp-RenderedText,.jp-OutputArea-output pre{font-size:1.03rem}
.cm-editor .cm-content{font-size:1.03rem}</style>"""))
import json, os
from pathlib import Path

from dotenv import load_dotenv
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
HAS_KEY = bool(os.getenv("OPENROUTER_API_KEY"))
LIVE_MODEL = os.getenv("AGENT_MODEL", "openrouter:openai/gpt-5.4-mini")  # live sample model (cache is openrouter:openai/gpt-5.4-mini)

SAMPLE_SIZE = 0  # >0 → run this many questions LIVE (needs a key + a 5-crate Neo4j graph). 0 = cache only.

CACHE = json.loads((ROOT/"data/eval/graph_vs_grep_cache.json").read_text())
GOLD  = json.loads((ROOT/"data/eval/gold.json").read_text())
K = CACHE.get("repeat", 1); R = CACHE["records"]; A = CACHE["adoption"]
print(f"cache: {len(R)} questions x k={K} runs | model {CACHE['model']} | live key available: {HAS_KEY}")

cache: 30 questions x k=5 runs | model openrouter:openai/gpt-5.4-mini | live key available: True


## Two agents, same brain

The only variable under test is the **tool surface**. Same model, `temperature=0`, same turn budget.
(Built in nb01 — shown here for reference; the cache below was produced by exactly these.)

```python
from context_workshop.graph.agent import build_graph_agent          # L3: graph traversal tools
graph_agent = build_graph_agent(MODEL, GraphTools(driver))

from pydantic_ai_backends import LocalBackend, create_console_toolset  # L1: grep/glob/read
grep_agent = Agent(MODEL, toolsets=[create_console_toolset(include_execute=False)], ...)
```

The graph agent has *curated tools over resolved structure*; the grep agent has *raw filesystem access*.
That asymmetry **is** the thing under test — "what a context layer enables vs raw source" — not a rig.

## The corpus — 30 questions, and gold that's *triangulated*, not graph-trusted

Each question is a **real task** — *"what depends on `X` I'm about to change?"* — not "where is `X` defined?".
The 30 split into **four classes we never aggregate**, because each tests a different thing and a *different
tool is supposed to win*:

| Class | n | A question like… | Should win | Why |
|---|---|---|---|---|
| **type-dependency** | 15 | "what public types break if I change `PostingList`?" | **graph** | relationships are *edges* — grep can't follow them |
| **collision** | 6 | "there are two `PostingList`s — which one do I use?" | **graph** | identity by canonical path; grep returns an ambiguous pile |
| **private internals** | 5 | "is there an *internal* helper for a posting list's compressed size?" | **grep** | the public rustdoc graph **can't see private items** — the honest boundary |
| **control** | 4 | "what non-owning view type borrows `PostingList`?" | **both** | a parity check — if the graph *lost* here, the eval would be rigged |

Two classes the graph is *built* for (type-dep, collision); one it **structurally can't answer** (private —
included on purpose); one is a sanity tie. Reporting them **separately** is the whole point — a single
aggregate would bury both the wins *and* the honest loss.

The gold is **mined from the graph → then verified against source**. The graph gives *candidate* answers;
we confirm each against the actual code before trusting it. Two steps — triangulation, not circularity.


In [3]:
from collections import Counter
print("classes:", dict(Counter(g["klass"] for g in GOLD)))
for name in ("col-pl-sparse", "td-encodedvectors", "pv-compressed-size"):
    g = next(x for x in GOLD if x["name"] == name)
    print(f"\n[{g['klass']} · expect {g['expect']}] {g['name']}")
    print(f"  Q:    {g['q']}")
    print(f"  gold: {g['anchors']}")

classes: {'typedep': 15, 'collision': 6, 'private': 5, 'control': 4}

[collision · expect graph] col-pl-sparse
  Q:    I'm storing sparse vectors and need the posting-list type built for them. There are multiple `PostingList` types here — which one do I use, and where is it defined?
  gold: [{'basename': 'posting_list.rs', 'line': 12, 'symbol': 'PostingList', 'crate': 'sparse'}]

[typedep · expect graph] td-encodedvectors
  Q:    I'm changing the `EncodedVectors` trait in `quantization`. What implements it (including anything outside the quantization crate)?
  gold: [{'symbol': 'EncodedVectorsU8', 'mode': 'named'}, {'symbol': 'QuantizedMultivectorStorage', 'mode': 'named', 'crate': 'segment'}]

[private · expect grep] pv-compressed-size
  Q:    I want a posting list's compressed size in bytes. Is there already an internal helper that computes the compressed size of one encoded chunk I can follow?
  gold: [{'basename': 'posting_list.rs', 'line': 72, 'symbol': 'get_compressed_size'}]


## A good question vs a bad one

The corpus lives or dies on question quality. A good question is a **real task** with **gold you can verify
against source**, scored **deterministically**. A bad one quietly needs an LLM judge — and then you're
measuring with a ruler that bends.

**✓ Good** — `td-storageoptions`: *"Which public Gridstore constructors take `StorageOptions`?"*
A real refactoring task; gold is a **named set**, each verified to exist in source; the check is mechanical —
did the answer name them, crate-guarded, with `file:line`? Yes/no, no opinion.

**✗ Bad** — *"Is `PostingList` well-designed?"*
No ground truth, so the only scorer is another model's vibe (non-deterministic) — and it doesn't even test
graph-vs-grep; it measures the *judge*.

The rule: **if you can't write the check as code against source, it isn't an eval question yet.**

## The scorer — deterministic, gold-anchored. **No LLM judge.**

Most eval tutorials reach for an LLM judge. We don't, and it's worth understanding *why*:

- **Cheap & fast** — no model calls to score; the whole eval re-runs for free.
- **Reproducible & unbiased** — the same answer always scores the same; no judge favoring confident prose.
- **It's the only ruler that can measure determinism.** An LLM judge is itself non-deterministic — using
  one to measure *reliability* (the result below) would pollute the very thing we're measuring.

The cost is discipline: you must say *exactly* what "correct" means. Here it's **multi-anchor matching** —
the answer must cite the gold symbol at its `file:line` (±1), crate-guarded so a near-line wrong-crate
hit can't be miscredited. **Unit-test the ruler before you trust the measurements:**

In [4]:
import re
def _line_near(text, basename, line):
    for m in re.finditer(re.escape(basename), text, re.IGNORECASE):
        pre = text[max(0, m.start()-18):m.start()].lower()
        if any(w in pre for w in ("not ","n't","isn","aren","rather than","instead of")):
            continue
        win = text[m.end():m.end()+60]
        md_ = re.match(r"[`'\")\]]*\s*[:#@]L?\s*(\d+)", win)
        if md_ and abs(int(md_.group(1))-line) <= 1:
            return True
        for wm in re.finditer(r"\bline(?:_start|s)?\b[^\d]{0,8}(\d+)", win, re.IGNORECASE):
            if abs(int(wm.group(1))-line) <= 1:
                return True
    return False
def _one(text, a):
    t = text or ""
    if a["symbol"].lower() not in t.lower(): return False
    if a.get("crate") and a["crate"].lower() not in t.lower(): return False   # collision guard
    if a.get("mode") == "named": return True
    return _line_near(t, a["basename"], a["line"])
def cites_all(text, anchors):           # the whole scorer: every gold anchor must be satisfied
    return all(_one(text, a) for a in anchors)

# --- trust the ruler first ---
S = [{"basename":"posting_list.rs","line":12,"symbol":"PostingList","crate":"sparse"}]
assert cites_all("sparse `PostingList` at posting_list.rs:12", S)                       # inline
assert cites_all("sparse PostingList — lib/sparse/.../posting_list.rs (line_start: 12)", S)  # graph format
assert not cites_all("PostingList at posting_list.rs:12", S)                            # crate missing
assert not cites_all("sparse PostingList is NOT at posting_list.rs:12", S)              # negated
assert not cites_all("PostingList for sparse at posting_list.rs:27", S)                 # wrong line
print("scorer self-test passed — now the numbers mean something")

scorer self-test passed — now the numbers mean something


## The run — `pydantic-evals`

A `Case` is **one question**. The **task** is the function that runs it — and it runs the *same* question on
**two agents** (a graph-only agent and a grep-only agent), returning *both* answers. The `Evaluator` then
scores each answer against the *same* gold. So one `Case` → two agents → one row with `graph_hit` + `grep_hit`.
That's the entire graph-vs-grep comparison: same question, same scorer, two toolsets.

```python
from pydantic_ai import Agent
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Evaluator

# Two agents, identical except their TOOLS — this is the whole experiment.
graph_agent = build_graph_agent(MODEL, graph_tools)             # A · graph traversal only
grep_agent  = Agent(MODEL, toolsets=[fs_tools], deps_type=Fs)   # B · grep / glob / read_file only

async def run_question(gold):                 # the TASK: one question -> BOTH agents
    graph_ans = (await graph_agent.run(gold["q"])).output            # A answers
    grep_ans  = (await grep_agent.run(gold["q"], deps=fs)).output    # B answers, same question
    return {"graph": graph_ans, "grep": grep_ans}                    # both go to the scorer ↓
    # (the full harness also runs the naive/nudged adoption arms here — same pattern, eval/harness.py)

class GoldScores(Evaluator):                  # scores EACH agent's answer against the same gold
    def evaluate(self, ctx):
        gold, out = ctx.inputs, ctx.output
        return {"graph_hit": cites_all(out["graph"], gold["anchors"]),
                "grep_hit":  cites_all(out["grep"],  gold["anchors"])}

dataset = Dataset(name="graph-vs-grep",
    cases=[Case(name=g["name"], inputs=g, metadata={"klass": g["klass"]}) for g in GOLD],
    evaluators=[GoldScores()])
report = await dataset.evaluate(run_question, max_concurrency=8, repeat=5)   # every case, ×5
```

`max_concurrency` parallelizes; `repeat=k` runs each question *k* times — which is what makes the
reliability metric (pass^k, below) possible. The committed cache below is one full `repeat=5` run. Want to
*see* it live? Set `SAMPLE_SIZE > 0` and run the next cell.


In [6]:
# OPTIONAL live taste — gated. Loads its OWN 5-crate graph into Neo4j (replaces nb01's), needs a key.
if SAMPLE_SIZE and HAS_KEY:
    import asyncio, random
    DOC = ROOT/"data/cache/cargo-target/doc"; CR = ["posting_list","sparse","segment","quantization","gridstore"]
    if all((DOC/f"{c}.json").exists() for c in CR):
        from context_workshop.graph import build_rust_graph, dedupe_edges, connect, load, GraphTools
        from context_workshop.graph.agent import build_graph_agent
        nodes, edges = [], []
        for c in CR:
            n,e = build_rust_graph(DOC/f"{c}.json"); nodes += n; edges += e
        keep = {n["qualified_name"] for n in nodes if n["qualified_name"].split("::",1)[0] in set(CR)}
        nodes = list({n["qualified_name"]: n for n in nodes if n["qualified_name"] in keep}.values())
        edges = [e for e in dedupe_edges(edges) if e.src in keep and e.dst in keep]
        drv = connect("neo4j://localhost:7687","neo4j","workshop123"); load(drv, nodes, edges, reset=True, assert_connected=False)
        gagent = build_graph_agent(LIVE_MODEL, GraphTools(drv))
        for g in random.sample(GOLD, SAMPLE_SIZE):
            res = await gagent.run(g["q"])
            print(f"  {g['name']:<20} graph hit={cites_all(res.output, g['anchors'])}")
        drv.close()
    else:
        print("Live sample needs the 5-crate rustdoc cache (data/cache/cargo-target/doc). Rendering cache only.")
else:
    print("Live sample off (SAMPLE_SIZE=0 or no key) — rendering the committed cache below.")

Live sample off (SAMPLE_SIZE=0 or no key) — rendering the committed cache below.


## Result 1 — single-shot accuracy: **a tie**

Per class, expected single-shot hits (the mean over the `k` runs). A strong model greps *well*.

In [7]:
print(f"{'class':<11}{'graph':>10}{'grep':>10}")
for k in ("typedep","collision","private","control"):
    rs = [r for r in R if r["klass"]==k]
    g = round(sum((r['graph']['hit'] or 0) for r in rs),1); p = round(sum((r['grep']['hit'] or 0) for r in rs),1)
    print(f"{k:<11}{f'{g}/{len(rs)}':>10}{f'{p}/{len(rs)}':>10}")
gt = round(sum((r['graph']['hit'] or 0) for r in R),1); pt = round(sum((r['grep']['hit'] or 0) for r in R),1)
print(f"{'TOTAL':<11}{f'{gt}/{len(R)}':>10}{f'{pt}/{len(R)}':>10}   <- basically even. We don't claim 'more correct.'")

class           graph      grep
typedep       13.6/15    9.0/15
collision       4.2/6     4.2/6
private           0/5     2.6/5
control         3.0/4     2.8/4
TOTAL         20.8/30   18.6/30   <- basically even. We don't claim 'more correct.'


## Result 2 — run it *k* times: **pass@k vs pass^k**

Same runs, two metrics, pointing opposite ways:

- **pass@k** = right *at least once* in `k` tries. Rewards retries — it *flatters the noisy method*.
- **pass^k** = right on **every** one of `k` runs. Rewards **determinism / reliability**.

The graph **looked the connections up** (computed once, deterministically, at Rung 1) → same answer every
run. Grep **re-derives the relationships from raw text each run** → a dice roll. This is the real win, and
it's the one single-shot accuracy hides.

In [7]:
patk = lambda side: sum(1 for r in R if (r[side]['hit'] or 0) > 0)
powk = lambda side: sum(1 for r in R if (r[side]['hit'] or 0) == 1.0)
print(f"all {len(R)} questions, k={K}:")
print(f"  pass@{K} (ever right):      graph {patk('graph')}/{len(R)}   grep {patk('grep')}/{len(R)}   <- ~tie (retries flatter grep)")
print(f"  pass^{K} (right EVERY run): graph {powk('graph')}/{len(R)}   grep {powk('grep')}/{len(R)}   <- the graph is deterministic; grep flips")
td = [r for r in R if r["klass"]=="typedep"]
gtd = sum(1 for r in td if (r['graph']['hit'] or 0)==1.0); ptd = sum(1 for r in td if (r['grep']['hit'] or 0)==1.0)
print(f"\n  type-dep only · pass^{K}:   graph {gtd}/{len(td)}   grep {ptd}/{len(td)}")

all 30 questions, k=5:
  pass@5 (ever right):      graph 22/30   grep 26/30   <- ~tie (retries flatter grep)
  pass^5 (right EVERY run): graph 19/30   grep 11/30   <- the graph is deterministic; grep flips

  type-dep only · pass^5:   graph 12/15   grep 6/15


## Result 3 — efficiency: same answers, **~3× fewer tokens**

Grep wades through whole files; the graph returns dense resolved rows.

In [8]:
def avg(side):
    xs = [r[side]["in_tok"] for r in R if r[side].get("in_tok") is not None]
    return round(sum(xs)/len(xs)) if xs else None
gt, pt = avg("graph"), avg("grep")
if gt and pt:
    print(f"avg input tokens / question:   graph {gt:,}   grep {pt:,}   ({round(pt/gt,1)}x cheaper)")

avg input tokens / question:   graph 7,444   grep 23,189   (3.1x cheaper)


## Result 4 — adoption: **the agent has to *use* the graph**

A third agent gets **both** graph and grep tools. We count which it actually calls. Naively it greps right
past the graph; shape the tools (name it `search`, ripgrep-shaped results) and it climbs — but never to 100%.
A perfect tool the agent ignores is worth nothing; this is the number that gates every result above.

In [9]:
def m(side):
    xs = [a[side]["adoption"] for a in A if a[side]["adoption"] is not None]
    return round(sum(xs)/len(xs),2) if xs else None
print(f"graph-tool adoption (share of code-lookups):   naive {m('naive')}  ->  nudged {m('nudged')}   (n={len(A)})")

graph-tool adoption (share of code-lookups):   naive 0.34  ->  nudged 0.72   (n=25)


## Watch it switch — naive vs nudged, live

The 34% → 72% above is the aggregate. Here's the single moment behind it: **same agent, same tools, same
question** — only the system prompt changes. Naive, the graph just sits there and the agent greps into the
`PostingList` collision. Nudged — name the tools for what they do, prefer the graph — and it resolves the
right `PostingList` by canonical path in two hops. *(This is the slide-13 hands-on.)*

In [10]:
# Same tools available to both; the SYSTEM PROMPT is the only change. Needs OPENROUTER_API_KEY + Neo4j up.
SWITCH_Q = ("I'm storing sparse vectors and need the posting-list type built for them. There are multiple "
            "`PostingList` types here — which one do I use, and where is it defined?")

if HAS_KEY:
    from context_workshop.graph import build_rust_graph, dedupe_edges, connect, load, GraphTools
    from context_workshop.eval.nudges import build_fallback_agent, tool_breakdown, adoption, FsDeps
    from pydantic_ai_backends import LocalBackend, create_console_toolset
    DOC = ROOT/"data/cache/cargo-target/doc"; LIB = ROOT/"data/raw_repos/qdrant/lib"
    CR4 = ["posting_list", "sparse", "gridstore", "quantization"]   # the committed 4-crate seed (no segment needed)
    if all((DOC/f"{c}.json").exists() for c in CR4) and LIB.exists():
        nodes, edges = [], []
        for c in CR4:
            n, e = build_rust_graph(DOC/f"{c}.json"); nodes += n; edges += e
        keep = {n["qualified_name"] for n in nodes if n["qualified_name"].split("::", 1)[0] in set(CR4)}
        nodes = list({n["qualified_name"]: n for n in nodes if n["qualified_name"] in keep}.values())
        edges = [e for e in dedupe_edges(edges) if e.src in keep and e.dst in keep]
        drv = connect("neo4j://localhost:7687", "neo4j", "workshop123")
        load(drv, nodes, edges, reset=True, assert_connected=False)
        gt, fs = GraphTools(drv), create_console_toolset(include_execute=False)
        deps = FsDeps(backend=LocalBackend(root_dir=str(LIB)))
        for label, nudge in [("naive ", False), ("nudged", True)]:
            agent = build_fallback_agent(LIVE_MODEL, gt, fs, nudge=nudge)
            res = await agent.run(SWITCH_Q, deps=deps)
            tb = tool_breakdown(res); share = adoption(tb)[0]
            tools = ", ".join(f"{k}x{v}" for k, v in tb.items()) or "(no tool calls)"
            print(f"[{label}] graph-adoption={share}   tools: {tools}")
            print(f"          {res.output.splitlines()[0][:96] if res.output else ''}")
        drv.close()
    else:
        print("Live switch needs the 4-crate seed (run scripts/setup_data.sh) + `docker compose up -d`.")
else:
    print("— EXAMPLE (no key/Docker). Set OPENROUTER_API_KEY + docker compose up -d to run it live —")
    print("naive  -> greps into the PostingList collision, picks one (maybe the wrong crate)")
    print("nudged -> search + get_symbol: resolves sparse::index::posting_list::PostingList, 2 hops")

— EXAMPLE (no key/Docker). Set OPENROUTER_API_KEY + docker compose up -d to run it live —
naive  -> greps into the PostingList collision, picks one (maybe the wrong crate)
nudged -> search + get_symbol: resolves sparse::index::posting_list::PostingList, 2 hops


## Build your own honest eval — the recipe (no LLM judge required)

1. **Real tasks, gold *triangulated*** — mine candidates, then verify against ground truth.
2. **A deterministic, anchored scorer — and unit-test it first.** Cheaper, reproducible, unbiased, and the
   only ruler that can measure reliability.
3. **`pydantic-evals`** — `Dataset` / `Case` / `Evaluator`, `max_concurrency`, and `repeat=k`.
4. **Report `pass^k`, not just `pass@1`** — single-shot hides whether the win is *reliable*.
5. **Per class, never aggregated. Name where it loses.** The honest losses are the roadmap (here: private internals → a v2 internals index).
6. **Measure adoption** — outcome quality is moot if the agent won't reach for the tool.

The verdict here: the graph's win was never per-shot accuracy. It's **determinism + efficiency** — *look it
up once* vs *figure it out every time* — and it only counts when the agent actually uses it.